# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026
### ⚙️ Before Running:
1. Datasets already added ✅
2. **Accelerator → GPU T4 x2** ✅
3. **Persistence → Files only** ✅
4. Run cells top to bottom — Steps 5+6 are combined to auto-start classifier after segmentation

## Step 0: Inspect Input Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=2):
    if depth > max_depth or not os.path.exists(path):
        return
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

explore('/kaggle/input', max_depth=2)

## Step 1: Clone / Pull Repo

In [ ]:
import os
if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    !cd /kaggle/working/DermaFusion-AI && git pull

%cd /kaggle/working/DermaFusion-AI
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
%pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — Settings > Accelerator > GPU T4 x2')

## Step 4: Link All Datasets → data/ folder

In [ ]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

search_roots = ['/kaggle/input/competitions']
datasets_dir = '/kaggle/input/datasets'
if os.path.exists(datasets_dir):
    for username in os.listdir(datasets_dir):
        user_path = os.path.join(datasets_dir, username)
        if os.path.isdir(user_path):
            search_roots.append(user_path)
search_roots.append('/kaggle/input')

def find_folder(keywords):
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for folder in os.listdir(root):
            full = os.path.join(root, folder)
            if os.path.isdir(full) and any(kw in folder.lower() for kw in keywords):
                return full
    return None

def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.exists(dst):
        print(f'✅ {label}: already linked')
        return
    if src:
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
    else:
        print(f'❌ {label}: NOT FOUND')

link(find_folder(['ham10000', 'skin-cancer-mnist']),               'ham10000',  'HAM10000')
link(find_folder(['isic-2019', 'isic2019', 'skin-lesion-images']), 'isic_2019', 'ISIC 2019')
link(find_folder(['siim-isic', 'melanoma-classification']),        'isic_2020', 'ISIC 2020')
link(find_folder(['isic-2024', 'skin-cancer-detection-3d']),       'isic_2024', 'ISIC 2024')

print('\n📁 data/ contents:', os.listdir(data_dir))

## Steps 5 + 6: Full Training Pipeline (Auto-Sequential)
**Both phases run automatically — no need to interact.**
- **Phase 1 (segmentation)** trains Swin-UNet for 25 epochs, saves `best_unet.pth`
- **Phase 2 (classifier)** starts automatically when Phase 1 finishes, loads the saved UNet
- Both phases save a **resume checkpoint** after each epoch — if the session times out, just re-run this cell to continue from where it stopped

In [ ]:
%cd /kaggle/working/DermaFusion-AI

print('=' * 60)
print('🚀 PHASE 1 — Segmentation Training (25 epochs)')
print('   Swin-Tiny UNet | SEG_BATCH_SIZE=8 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log

print('\n')
print('=' * 60)
print('🚀 PHASE 2 — Classifier Training (25 epochs)')
print('   EVA-02 Large + ConvNeXt V2 | BATCH_SIZE=2 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log

print('\n✅ Both phases complete! Run Step 7 to save weights.')

## Step 7: Save Weights to Output Tab

In [ ]:
import shutil, os
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            dst = f'/kaggle/working/{f}'
            shutil.copy(os.path.join(weights_src, f), dst)
            print(f'✅ {f}  ({os.path.getsize(dst)/1e6:.0f} MB)')
    print('\n📦 Download from the Output tab on the right.')
else:
    print('❌ No weights folder — did training complete?')

## Step 8: Full Evaluation

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete!')